# Resolución de la consigna: series temporales de BodegaAI

Este notebook responde la consigna de punta a punta usando **dos series horarias reales**:

- `alb-prod-requests-2026-07.csv`: requests totales del ALB durante julio de 2026.
- `bodegaai-production-pos-api-request-count-per-target-last-30-days-hourly.csv`: requests por target del POS API durante los últimos 30 días disponibles.

La idea central es simple: primero entendemos cada serie, luego evaluamos estacionariedad, ajustamos modelos SARIMA, diagnosticamos residuos, pronosticamos y finalmente usamos VAR sobre ambas series alineadas. Ojo: las series tienen **rangos de fechas distintos**, así que el análisis conjunto solo usa el período superpuesto.


In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from matplotlib.ticker import FuncFormatter, ScalarFormatter
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import acf, acovf, adfuller, kpss, grangercausalitytests
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.api import VAR

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:,.2f}".format)

COUNT_FORMATTER = FuncFormatter(lambda x, pos: f"{x:,.0f}")

def format_count_axis(ax):
    ax.yaxis.set_major_formatter(COUNT_FORMATTER)
    return ax


def clean_count_table(df):
    return df.copy().style.format(precision=2, thousands=",", decimal=".")


## 1. Carga y preparación

Trabajamos con frecuencia horaria. Para que los modelos no inventen huecos temporales, cada serie se reindexa a `1H` y se interpolan faltantes si aparecieran. En estas series los conteos son grandes; por eso las tablas y ejes se formatean sin notación científica.


In [ ]:
ALB_PATH = "data/alb-prod-requests-2026-07.csv"
POS_PATH = "data/bodegaai-production-pos-api-request-count-per-target-last-30-days-hourly.csv"


def load_hourly_series(path, value_col, name):
    df = pd.read_csv(path)
    df["period_start_utc"] = pd.to_datetime(df["period_start_utc"], utc=True)
    series = (
        df.sort_values("period_start_utc")
          .set_index("period_start_utc")[value_col]
          .astype(float)
          .rename(name)
    )
    full_index = pd.date_range(series.index.min(), series.index.max(), freq="h", tz="UTC")
    series = series.reindex(full_index).interpolate("time")
    series.index.name = "fecha_hora_utc"
    return series

alb = load_hourly_series(ALB_PATH, "requests", "requests_alb")
pos = load_hourly_series(POS_PATH, "value", "requests_pos_api")

summary = pd.DataFrame({
    "serie": ["ALB total", "POS API por target"],
    "inicio": [alb.index.min(), pos.index.min()],
    "fin": [alb.index.max(), pos.index.max()],
    "observaciones": [len(alb), len(pos)],
    "faltantes": [int(alb.isna().sum()), int(pos.isna().sum())],
    "mínimo": [alb.min(), pos.min()],
    "media": [alb.mean(), pos.mean()],
    "máximo": [alb.max(), pos.max()],
})
clean_count_table(summary)


## 2. Series originales y necesidad de diferenciar

Una serie estacionaria mantiene media, varianza y autocorrelación relativamente estables en el tiempo. En series horarias de tráfico suele aparecer una estructura diaria: horas de mayor y menor uso que se repiten cada 24 observaciones. Si esa estructura domina, conviene probar **diferenciación estacional** (`y_t - y_{t-24}`) y, si todavía queda tendencia local, una diferencia regular (`y_t - y_{t-1}`).


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=False)
for ax, series, title in [
    (axes[0], alb, "Requests ALB total"),
    (axes[1], pos, "Requests POS API por target"),
]:
    ax.plot(series.index, series.values, linewidth=1)
    ax.set_title(title)
    ax.set_ylabel("requests")
    format_count_axis(ax)
plt.tight_layout()
plt.show()


In [ ]:
def transformed_versions(series):
    return pd.DataFrame({
        "original": series,
        "diff_1": series.diff(1),
        "diff_24": series.diff(24),
        "diff_1_y_24": series.diff(24).diff(1),
    }).dropna()

fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=False)
for row, (series, title) in enumerate([(alb, "ALB"), (pos, "POS API")]):
    transformed = transformed_versions(series)
    axes[row, 0].plot(transformed.index, transformed["diff_24"], linewidth=0.8)
    axes[row, 0].set_title(f"{title}: diferencia estacional 24h")
    axes[row, 1].plot(transformed.index, transformed["diff_1_y_24"], linewidth=0.8)
    axes[row, 1].set_title(f"{title}: diferencia 24h + diferencia 1h")
    for ax in axes[row]:
        ax.axhline(0, color="black", linewidth=0.8)
        format_count_axis(ax)
plt.tight_layout()
plt.show()


## 3. FAS, FAC y FACP

- **FAS**: función de autocovarianza muestral. Mide dependencia en unidades originales.
- **FAC / ACF**: autocorrelación. Normaliza la autocovarianza entre -1 y 1.
- **FACP / PACF**: autocorrelación parcial. Mide la relación con un rezago descontando los rezagos intermedios.

Si aparecen picos en los rezagos 24, 48 o 72, hay evidencia visual de estacionalidad diaria. Si la FAC cae lentamente, la serie probablemente necesita diferenciación.


In [ ]:
def plot_fas_fac_facp(series, title, lags=72):
    clean = series.dropna()
    fas_values = acovf(clean, fft=True, nlag=lags)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    axes[0].stem(range(len(fas_values)), fas_values, basefmt=" ")
    axes[0].set_title(f"FAS - {title}")
    axes[0].set_xlabel("rezago")
    format_count_axis(axes[0])

    plot_acf(clean, lags=lags, ax=axes[1], zero=False)
    axes[1].set_title(f"FAC - {title}")

    plot_pacf(clean, lags=lags, ax=axes[2], zero=False, method="ywm")
    axes[2].set_title(f"FACP - {title}")

    for ax in axes:
        ax.axvline(24, color="tab:red", linestyle="--", linewidth=1, alpha=0.7)
        ax.axvline(48, color="tab:red", linestyle="--", linewidth=1, alpha=0.5)
        ax.axvline(72, color="tab:red", linestyle="--", linewidth=1, alpha=0.4)
    plt.tight_layout()
    plt.show()

plot_fas_fac_facp(alb, "ALB original")
plot_fas_fac_facp(pos, "POS API original")


## 4. Pruebas de raíces unitarias

Usamos dos pruebas complementarias:

- **ADF**: hipótesis nula = la serie tiene raíz unitaria, es decir, no es estacionaria.
- **KPSS**: hipótesis nula = la serie es estacionaria alrededor de una constante o tendencia.

La lectura conjunta evita una decisión mecánica. Si ADF no rechaza y KPSS rechaza, la evidencia apunta a no estacionariedad. Si luego de diferenciar ADF rechaza y KPSS no rechaza, la transformación es razonable.


In [ ]:
def unit_root_tests(series, name):
    rows = []
    versions = {
        "original": series.dropna(),
        "diff_1": series.diff(1).dropna(),
        "diff_24": series.diff(24).dropna(),
        "diff_1_y_24": series.diff(24).diff(1).dropna(),
    }
    for version_name, values in versions.items():
        adf_stat, adf_p, _, _, _, _ = adfuller(values, autolag="AIC")
        try:
            kpss_stat, kpss_p, _, _ = kpss(values, regression="c", nlags="auto")
        except Exception:
            kpss_stat, kpss_p = np.nan, np.nan
        rows.append({
            "serie": name,
            "versión": version_name,
            "ADF estadístico": adf_stat,
            "ADF p-valor": adf_p,
            "KPSS estadístico": kpss_stat,
            "KPSS p-valor": kpss_p,
        })
    return pd.DataFrame(rows)

unit_tests = pd.concat([
    unit_root_tests(alb, "ALB"),
    unit_root_tests(pos, "POS API"),
], ignore_index=True)
clean_count_table(unit_tests)


## 5. SARIMA: comparación de órdenes

Como los datos son horarios y muestran patrón diario, se usa estacionalidad `s = 24`. Para no sobreajustar ni gastar cómputo innecesario, se prueba una grilla chica de candidatos. El criterio principal de comparación es AIC/BIC dentro del set de entrenamiento, y luego validamos con métricas fuera de muestra.


In [ ]:
def train_test_split_series(series, test_size=48):
    train = series.iloc[:-test_size]
    test = series.iloc[-test_size:]
    return train, test


def manual_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    errors = y_true - y_pred
    mae = np.mean(np.abs(errors))
    rmse = np.sqrt(np.mean(errors ** 2))
    denom = np.where(y_true == 0, np.nan, y_true)
    mape = np.nanmean(np.abs(errors / denom)) * 100
    return {"MAE": mae, "RMSE": rmse, "MAPE_%": mape}

# Grilla chica: combina diferencia regular/estacional con pocos órdenes ARMA.
SARIMA_CANDIDATES = [
    {"order": (1, 0, 1), "seasonal_order": (1, 0, 1, 24)},
    {"order": (1, 1, 1), "seasonal_order": (1, 0, 1, 24)},
    {"order": (1, 0, 1), "seasonal_order": (1, 1, 1, 24)},
    {"order": (1, 1, 1), "seasonal_order": (1, 1, 1, 24)},
]


def fit_sarima_candidates(series, name, test_size=48):
    train, test = train_test_split_series(series, test_size=test_size)
    rows = []
    fitted_models = {}
    for candidate in SARIMA_CANDIDATES:
        order = candidate["order"]
        seasonal_order = candidate["seasonal_order"]
        label = f"SARIMA{order}x{seasonal_order}"
        try:
            model = SARIMAX(
                train,
                order=order,
                seasonal_order=seasonal_order,
                enforce_stationarity=False,
                enforce_invertibility=False,
            )
            result = model.fit(disp=False, maxiter=80)
            forecast = result.get_forecast(steps=len(test)).predicted_mean
            metrics = manual_metrics(test.values, forecast.values)
            rows.append({
                "serie": name,
                "modelo": label,
                "AIC": result.aic,
                "BIC": result.bic,
                "MAE": metrics["MAE"],
                "RMSE": metrics["RMSE"],
                "MAPE_%": metrics["MAPE_%"],
            })
            fitted_models[label] = result
        except Exception as exc:
            rows.append({
                "serie": name,
                "modelo": label,
                "AIC": np.nan,
                "BIC": np.nan,
                "MAE": np.nan,
                "RMSE": np.nan,
                "MAPE_%": np.nan,
            })
    comparison = pd.DataFrame(rows).sort_values(["MAPE_%", "AIC"], na_position="last")
    best_label = comparison.iloc[0]["modelo"]
    return comparison, fitted_models[best_label], best_label, train, test

alb_comparison, alb_best, alb_best_label, alb_train, alb_test = fit_sarima_candidates(alb, "ALB")
pos_comparison, pos_best, pos_best_label, pos_train, pos_test = fit_sarima_candidates(pos, "POS API")

sarima_comparison = pd.concat([alb_comparison, pos_comparison], ignore_index=True)
clean_count_table(sarima_comparison)


In [ ]:
def sarima_parameter_table(result, serie, modelo):
    conf = result.conf_int()
    table = pd.DataFrame({
        "serie": serie,
        "modelo": modelo,
        "parámetro": result.params.index,
        "coeficiente": result.params.values,
        "error_std": result.bse.values,
        "z": result.tvalues.values,
        "p_valor": result.pvalues.values,
        "ic_95_inf": conf.iloc[:, 0].values,
        "ic_95_sup": conf.iloc[:, 1].values,
    })
    return table

sarima_parameters = pd.concat([
    sarima_parameter_table(alb_best, "ALB", alb_best_label),
    sarima_parameter_table(pos_best, "POS API", pos_best_label),
], ignore_index=True)

sarima_selected = pd.DataFrame([
    {"serie": "ALB", "modelo_seleccionado": alb_best_label, "AIC": alb_best.aic, "BIC": alb_best.bic},
    {"serie": "POS API", "modelo_seleccionado": pos_best_label, "AIC": pos_best.aic, "BIC": pos_best.bic},
])

display(clean_count_table(sarima_selected))
display(clean_count_table(sarima_parameters))

## 6. Train/test y métricas manuales

Separamos las últimas 48 horas como testing. Las métricas se calculan manualmente:

- **MAE**: error absoluto promedio.
- **RMSE**: penaliza más los errores grandes.
- **MAPE**: error porcentual promedio; es interpretable si no hay valores cercanos a cero.


In [ ]:
def plot_test_forecast(train, test, result, title):
    pred = result.get_forecast(steps=len(test)).predicted_mean
    metrics = manual_metrics(test.values, pred.values)
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(train.index[-7*24:], train.iloc[-7*24:], label="train reciente", linewidth=1)
    ax.plot(test.index, test, label="test real", linewidth=1.5)
    ax.plot(test.index, pred, label="pronóstico test", linewidth=1.5)
    ax.set_title(f"{title} | MAE={metrics['MAE']:,.0f} RMSE={metrics['RMSE']:,.0f} MAPE={metrics['MAPE_%']:.2f}%")
    ax.legend()
    format_count_axis(ax)
    plt.tight_layout()
    plt.show()
    return metrics

alb_metrics = plot_test_forecast(alb_train, alb_test, alb_best, "ALB")
pos_metrics = plot_test_forecast(pos_train, pos_test, pos_best, "POS API")
clean_count_table(pd.DataFrame([{"serie": "ALB", **alb_metrics}, {"serie": "POS API", **pos_metrics}]))


## 7. Comparación con otros modelos estimados

La comparación relevante no es solo “menor AIC”: también importa cómo predice el tramo de test. En series con estacionalidad clara, los modelos con componente estacional suelen ganar porque capturan el ciclo diario que un ARIMA no estacional trataría como ruido persistente.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
sns.barplot(data=sarima_comparison, x="modelo", y="MAPE_%", hue="serie", ax=ax)
ax.set_title("Comparación SARIMA por MAPE en test")
ax.set_xlabel("modelo")
ax.set_ylabel("MAPE (%)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()


## 8. Diagnóstico de residuos

Un buen modelo debería dejar residuos aproximadamente centrados en cero, sin autocorrelación fuerte. Para eso revisamos:

- Serie temporal de residuos.
- Histograma.
- FAC de residuos.
- Test de Ljung-Box: hipótesis nula = no hay autocorrelación conjunta hasta el rezago evaluado.


In [ ]:
def residual_diagnostics(result, title, lags=48):
    resid = pd.Series(result.resid).dropna()
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(resid.index, resid.values, linewidth=0.8)
    axes[0].axhline(0, color="black", linewidth=0.8)
    axes[0].set_title(f"Residuos - {title}")
    format_count_axis(axes[0])

    sns.histplot(resid, bins=30, kde=True, ax=axes[1])
    axes[1].set_title(f"Distribución - {title}")
    format_count_axis(axes[1])

    plot_acf(resid, lags=lags, ax=axes[2], zero=False)
    axes[2].set_title(f"FAC residuos - {title}")
    plt.tight_layout()
    plt.show()

    lb = acorr_ljungbox(resid, lags=[12, 24, 48], return_df=True)
    return lb.reset_index().rename(columns={"index": "rezago", "lb_stat": "Ljung-Box", "lb_pvalue": "p-valor"})

alb_lb = residual_diagnostics(alb_best, "ALB")
pos_lb = residual_diagnostics(pos_best, "POS API")
clean_count_table(pd.concat([alb_lb.assign(serie="ALB"), pos_lb.assign(serie="POS API")], ignore_index=True))


## 9. Pronóstico SARIMA

La ventana elegida es de 48 horas porque conserva una escala razonable para series horarias y evita extrapolar demasiado lejos con muestras pequeñas. Más horizonte implica más incertidumbre, especialmente cuando el rango histórico es de apenas 30/31 días.


In [ ]:
def refit_and_forecast(series, best_label, steps=48):
    candidate = None
    for item in SARIMA_CANDIDATES:
        label = f"SARIMA{item['order']}x{item['seasonal_order']}"
        if label == best_label:
            candidate = item
            break
    model = SARIMAX(
        series,
        order=candidate["order"],
        seasonal_order=candidate["seasonal_order"],
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    result = model.fit(disp=False, maxiter=80)
    forecast = result.get_forecast(steps=steps)
    mean = forecast.predicted_mean
    interval = forecast.conf_int()
    return result, mean, interval

alb_final, alb_forecast, alb_interval = refit_and_forecast(alb, alb_best_label)
pos_final, pos_forecast, pos_interval = refit_and_forecast(pos, pos_best_label)


def plot_future_forecast(series, mean, interval, title):
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(series.index[-7*24:], series.iloc[-7*24:], label="histórico reciente")
    ax.plot(mean.index, mean, label="pronóstico", color="tab:orange")
    ax.fill_between(mean.index, interval.iloc[:, 0], interval.iloc[:, 1], color="tab:orange", alpha=0.2, label="IC 95%")
    ax.set_title(title)
    ax.legend()
    format_count_axis(ax)
    plt.tight_layout()
    plt.show()

plot_future_forecast(alb, alb_forecast, alb_interval, "Pronóstico 48h - ALB")
plot_future_forecast(pos, pos_forecast, pos_interval, "Pronóstico 48h - POS API")


## 10. VAR sobre series alineadas y transformadas

Para VAR necesitamos ambas series en las mismas fechas. Como los archivos cubren rangos diferentes, se toma la intersección temporal. Además, se aplica `log1p` y diferencia estacional de 24 horas para trabajar con una dinámica más estable. Esta transformación cambia la interpretación: el VAR modela variaciones logarítmicas estacionales, no niveles absolutos de requests.


In [ ]:
aligned = pd.concat([alb, pos], axis=1, join="inner").dropna()
aligned_log = np.log1p(aligned)
var_data = aligned_log.diff(24).dropna()

print("Rango ALB:", alb.index.min(), "a", alb.index.max())
print("Rango POS API:", pos.index.min(), "a", pos.index.max())
print("Rango común usado en VAR:", aligned.index.min(), "a", aligned.index.max())
print("Observaciones VAR después de transformar:", len(var_data))

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
for ax, col in zip(axes, var_data.columns):
    ax.plot(var_data.index, var_data[col], linewidth=0.8)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(f"VAR input: log1p({col}).diff(24)")
plt.tight_layout()
plt.show()


In [ ]:
var_train = var_data.iloc[:-48]
var_test = var_data.iloc[-48:]
var_model = VAR(var_train)
selection = var_model.select_order(maxlags=24)

selection_table = pd.DataFrame(selection.ics)
selection_table.index.name = "rezagos"
display(clean_count_table(selection_table.reset_index()))

selected_lag = selection.selected_orders.get("aic")
if selected_lag is None or selected_lag < 1:
    selected_lag = 1
selected_lag = int(min(selected_lag, 24))
print("Rezagos VAR seleccionados por AIC:", selected_lag)

var_result = var_model.fit(selected_lag)

var_params = var_result.params.reset_index().rename(columns={"index": "parámetro"})
var_pvalues = var_result.pvalues.reset_index().rename(columns={"index": "parámetro"})
var_params_long = var_params.melt(id_vars="parámetro", var_name="ecuación", value_name="coeficiente")
var_pvalues_long = var_pvalues.melt(id_vars="parámetro", var_name="ecuación", value_name="p_valor")
var_coefficients = var_params_long.merge(var_pvalues_long, on=["parámetro", "ecuación"])
display(clean_count_table(var_coefficients))

var_forecast_values = var_result.forecast(var_train.values[-selected_lag:], steps=len(var_test))
var_forecast = pd.DataFrame(var_forecast_values, index=var_test.index, columns=var_test.columns)

var_metrics = []
for col in var_test.columns:
    metrics = manual_metrics(var_test[col].values, var_forecast[col].values)
    var_metrics.append({"serie_transformada": col, **metrics})
clean_count_table(pd.DataFrame(var_metrics))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
for ax, col in zip(axes, var_test.columns):
    ax.plot(var_test.index, var_test[col], label="test real", linewidth=1.5)
    ax.plot(var_forecast.index, var_forecast[col], label="VAR forecast", linewidth=1.5)
    ax.set_title(f"VAR forecast sobre serie transformada: {col}")
    ax.legend()
plt.tight_layout()
plt.show()


## 11. Impulso-respuesta y causalidad de Granger

La función impulso-respuesta muestra cómo responde una variable del VAR ante un shock en otra variable. La causalidad de Granger no prueba causalidad económica real; prueba si los rezagos de una serie ayudan a predecir la otra, dado el modelo y la muestra disponible.


In [ ]:
irf = var_result.irf(24)
irf.plot(orth=False)
plt.suptitle("Funciones impulso-respuesta VAR", y=1.02)
plt.tight_layout()
plt.show()

causality_rows = []
for caused in var_data.columns:
    causing = [col for col in var_data.columns if col != caused]
    test = var_result.test_causality(caused=caused, causing=causing, kind="f")
    causality_rows.append({
        "prueba": "VAR ajustado",
        "variable_causada": caused,
        "variable_causante": ", ".join(causing),
        "estadístico_F": test.test_statistic,
        "p_valor": test.pvalue,
        "decisión_5pct": "rechaza H0" if test.pvalue < 0.05 else "no rechaza H0",
    })

def granger_table(data, caused, causing, lags=(6, 12, 24)):
    # statsmodels espera primero la variable causada y luego la causante.
    result = grangercausalitytests(data[[caused, causing]], maxlag=list(lags), verbose=False)
    rows = []
    for lag in lags:
        f_stat, p_value, _, _ = result[lag][0]["ssr_ftest"]
        rows.append({
            "prueba": "Granger bivariada",
            "variable_causada": caused,
            "variable_causante": causing,
            "rezagos": lag,
            "estadístico_F": f_stat,
            "p_valor": p_value,
            "decisión_5pct": "rechaza H0" if p_value < 0.05 else "no rechaza H0",
        })
    return rows

causality_rows.extend(granger_table(var_data, "requests_alb", "requests_pos_api"))
causality_rows.extend(granger_table(var_data, "requests_pos_api", "requests_alb"))

causality_results = pd.DataFrame(causality_rows)
display(clean_count_table(causality_results))

## 12. Conclusión y comparación SARIMA cuando hay estacionalidad

Ambas series son horarias y presentan señales compatibles con estacionalidad diaria: picos en FAC/FACP alrededor de 24 horas y patrones repetidos en los gráficos originales. Por eso SARIMA con período estacional `s = 24` es una representación más adecuada que ignorar la estacionalidad.

Lectura final:

1. **Estacionariedad**: las series en niveles no deberían tratarse como estacionarias sin verificación. La diferenciación estacional y/o regular mejora la estabilidad.
2. **SARIMA**: permite modelar dependencia de corto plazo y ciclo diario. La elección final debe mirar AIC/BIC, métricas en test y residuos.
3. **Diagnóstico**: si Ljung-Box rechaza fuertemente, todavía queda autocorrelación sin explicar; eso limita la confianza del pronóstico.
4. **VAR**: aporta una mirada conjunta, pero solo sobre el tramo temporal común y sobre series transformadas. Sus resultados deben interpretarse como evidencia predictiva, no como causalidad de negocio definitiva.
5. **Caveat principal**: la muestra es corta —aprox. 30/31 días— y los rangos no coinciden exactamente. Esto alcanza para una práctica académica, pero no para una decisión productiva robusta sin más historia, validación rolling y control de eventos externos.
